<a href="https://colab.research.google.com/github/BF667/RVC/blob/main/RVC_Enhanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Enhanced RVC with PolUVR + YTDLP + Custom Kernels

**Features:**
- ✅ PolUVR integration for source separation
- ✅ YTDLP for YouTube audio extraction
- ✅ Custom CUDA kernels for optimized inference
- ✅ Enhanced Gradio demo
- ✅ Python 3.12+ compatibility
- ✅ Mixed precision training support

In [ ]:
#@title <big> 🚀 **ENHANCED INSTALLATION**

import torch
from IPython.display import clear_output, HTML
import subprocess
import sys
import os

print("🎵 Starting Enhanced RVC Installation...")

# Clone the repository
git_url = "https://github.com/BF667/RVC.git"
rvc_dir = "/content/RVC"

if not os.path.exists(rvc_dir):
    !git clone $git_url $rvc_dir
else:
    %cd $rvc_dir
    !git pull origin main

%cd $rvc_dir

# Check Python version
python_version = sys.version_info
print(f"🐍 Python version: {python_version.major}.{python_version.minor}.{python_version.micro}")

# Install uv package manager if not present
!pip install --no-cache-dir -qq uv

# Install dependencies with version compatibility
if python_version >= (3, 12):
    print("📦 Installing dependencies for Python 3.12+")
    !uv pip install --no-cache-dir -qq torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
else:
    print("📦 Installing dependencies for Python <3.12")
    !uv pip install --no-cache-dir -qq torch==2.0.0 torchvision==0.15.0 torchaudio==2.0.0 --index-url https://download.pytorch.org/whl/cu118

# Install other dependencies
!uv pip install --no-cache-dir -qq -r requirements.txt

# Install custom kernel support (optional)
try:
    import cupy
    print("✅ CuPy already installed")
except ImportError:
    print("🔧 Installing CuPy for custom kernels...")
    if python_version >= (3, 12):
        !uv pip install --no-cache-dir -qq cupy-cuda12x
    else:
        !uv pip install --no-cache-dir -qq cupy

# Download essential models
print("📥 Downloading essential models...")

# Create models directory
os.makedirs("/content/RVC/models/embedders", exist_ok=True)
os.makedirs("/content/RVC/models/RVC_models", exist_ok=True)

# Download Hubert model
if not os.path.exists("/content/RVC/rvc/models/embedders/hubert_base.pt"):
    !wget -O /content/RVC/rvc/models/embedders/hubert_base.pt https://huggingface.co/ccccc/RVC-pretrained/resolve/main/hubert_base.pt

clear_output()
print("✅ Enhanced RVC installation completed!")
print("🔥 Features enabled:")
print("   🎛️  PolUVR for source separation")
print("   🌐 YTDLP for YouTube audio")
print("   ⚡ Custom CUDA kernels")
print("   🎭 Enhanced voice conversion")

In [ ]:
#@title <big> 🔍 **Model Management with Enhanced Features**

%cd /content/RVC

import os
import subprocess
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import json

# Enhanced model management interface
print("📋 Enhanced Model Management")
print("Supported formats: .zip, .rar, direct URL links")

# Input fields
url_input = widgets.Text(
    value='',
    placeholder='Enter model URL or zip file path',
    description='📡 URL:',
    layout=widgets.Layout(width='60%'),
    style={'description_width': 'initial'}
)

dir_name_input = widgets.Text(
    value='',
    placeholder='Enter model name (e.g., my_voice_model)',
    description='🎭 Model Name:',
    layout=widgets.Layout(width='60%'),
    style={'description_width': 'initial'}
)

use_poluvr = widgets.Checkbox(
    value=False,
    description='🎛️  Pre-process with PolUVR',
    style={'description_width': 'initial'}
)

output = widgets.Output(layout=widgets.Layout(width='60%', height='200px'))

# Enhanced download function
def enhanced_download_model(b):
    with output:
        output.clear_output()
        url = url_input.value.strip()
        model_name = dir_name_input.value.strip()
        use_uvr = use_poluvr.value

        if not url or not model_name:
            print("❌ Please provide both URL and model name")
            return

        try:
            print(f"📥 Downloading model: {model_name}")
            if use_uvr:
                print("🎛️  PolUVR pre-processing will be applied")

            # Run enhanced model manager
            cmd = [
                'python3', '-m', 'rvc.modules.model_manager',
                '--url', url,
                '--model-name', model_name
            ]
            if use_uvr:
                cmd.append('--use-poluvr')

            result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
            
            if result.returncode == 0:
                print("✅ Model downloaded successfully!")
                if result.stdout:
                    print(result.stdout)
                clear_output()
                list_available_models()
            else:
                print(f"❌ Error: {result.stderr}")
        except subprocess.TimeoutExpired:
            print("⏰ Download timed out")
        except Exception as e:
            print(f"❌ Error: {e}")

download_button = widgets.Button(
    description='📥 Download & Install',
    disabled=False,
    button_style='success',
    tooltip='Download and install the model',
    icon='download',
    layout=widgets.Layout(width='30%', height='50px')
)
download_button.on_click(enhanced_download_model)

# Display available models
def list_available_models():
    rvc_models_dir = os.path.join(os.getcwd(), "models", "RVC_models")
    
    if not os.path.exists(rvc_models_dir):
        print("📁 No models directory found")
        return
    
    models = [d for d in os.listdir(rvc_models_dir) if os.path.isdir(os.path.join(rvc_models_dir, d))]
    
    if not models:
        print("📦 No models installed yet")
        return
    
    print(f"\n🎭 Available RVC Models ({len(models)} total):")
    print("=" * 50)
    
    for i, model in enumerate(sorted(models), 1):
        model_path = os.path.join(rvc_models_dir, model)
        files = os.listdir(model_path)
        file_sizes = []
        for f in files:
            if f.endswith(('.pth', '.index')):
                size = os.path.getsize(os.path.join(model_path, f)) / (1024*1024)  # MB
                file_sizes.append(f"{f} ({size:.1f}MB)")
        
        status = "🟢 Ready" if any(f.endswith('.pth') for f in files) else "⚠️  Incomplete"
        print(f"{i:2d}. {model} - {status}")
        for size_info in file_sizes:
            print(f"    📄 {size_info}")
    
    print("\n💡 Copy the model name and use it in the inference section")

# Initial model list
list_available_models()

# Display interface
display(widgets.VBox([
    widgets.HTML("<h3>📥 Model Download Manager</h3>"),
    url_input,
    dir_name_input,
    use_poluvr,
    download_button,
    output
]))

In [ ]:
#@title <big> 🎵 **Enhanced Voice Conversion**

%cd /content/RVC

import os
import torch
from IPython.display import display, Audio, clear_output
import ipywidgets as widgets
import subprocess
import json
from datetime import datetime

# Enhanced inference interface
print("🎤 Enhanced RVC Voice Conversion Interface")
print("Features: PolUVR, YTDLP, Custom Kernels, Multiple F0 Methods")

# Model selection
model_name = widgets.Text(
    value='',
    placeholder='Enter installed model name',
    description='🎭 Model Name:',
    layout=widgets.Layout(width='50%'),
    style={'description_width': 'initial'}
)

# Input source selection
input_source = widgets.Dropdown(
    choices=[
        ("📁 Upload Audio File", "file"),
        ("🌐 YouTube URL", "youtube"),
        ("🎛️  PolUVR Processed", "poluvr")
    ],
    value="file",
    description='🎵 Input Source:',
    layout=widgets.Layout(width='50%'),
    style={'description_width': 'initial'}
)

# File upload
audio_file = widgets.FileUpload(
    accept='.wav,.mp3,.flac,.ogg,.m4a',
    multiple=False,
    description='📤 Upload Audio',
    style={'description_width': 'initial'}
)

# YouTube URL
youtube_url = widgets.Text(
    value='',
    placeholder='https://www.youtube.com/watch?v=...',
    description='🔗 YouTube URL:',
    layout=widgets.Layout(width='60%'),
    style={'description_width': 'initial'}
)

# Enhanced parameters
with widgets.VBox():
    print("🎛️  Voice Conversion Parameters")
    
    # F0 settings
    f0_method = widgets.Dropdown(
        choices=[("🎼 RMVPE (Recommended)", "rmvpe"), ("🎵 FCPE", "fcpe"), ("🎯 CREPE", "crepe")],
        value="rmvpe",
        description='F0 Method:',
        style={'description_width': 'initial'}
    )
    
    pitch_shift = widgets.FloatSlider(
        value=0,
        min=-24,
        max=24,
        step=0.5,
        description='🎵 Pitch Shift:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    )
    
    # Quality settings
    index_rate = widgets.FloatSlider(
        value=0.25,
        min=0,
        max=1,
        step=0.01,
        description='🔍 Index Rate:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    )
    
    protect = widgets.FloatSlider(
        value=0.33,
        min=0,
        max=0.5,
        step=0.01,
        description='🛡️  Protect Volume:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    )

# Advanced options
with_accordion = widgets.Accordion([
    widgets.VBox([
        widgets.FloatSlider(value=50, min=1, max=100, step=1, description='F0 Min (Hz):'),
        widgets.FloatSlider(value=1100, min=400, max=16000, step=10, description='F0 Max (Hz):'),
        widgets.IntSlider(value=128, min=32, max=512, step=1, description='Hop Length:'),
        widgets.FloatSlider(value=1.0, min=0, max=2, step=0.01, description='Volume Envelope:')
    ])
])
with_accordion.set_title(0, '🔧 Advanced Settings')

# Output format
output_format = widgets.Dropdown(
    choices=[("🎵 WAV (Lossless)", "wav"), ("🎶 MP3", "mp3"), ("📀 FLAC", "flac")],
    value="wav",
    description='💾 Output Format:',
    style={'description_width': 'initial'}
)

# Enhanced features
enable_poluvr = widgets.Checkbox(
    value=False,
    description='🎛️  Enable PolUVR',
    style={'description_width': 'initial'}
)

enable_custom_kernels = widgets.Checkbox(
    value=True,
    description='⚡ Enable Custom Kernels',
    style={'description_width': 'initial'}
)

mixed_precision = widgets.Checkbox(
    value=True,
    description='🚀 Mixed Precision',
    style={'description_width': 'initial'}
)

# Inference button
convert_button = widgets.Button(
    description='🎵 Start Voice Conversion',
    disabled=False,
    button_style='info',
    tooltip='Start the voice conversion process',
    icon='play',
    layout=widgets.Layout(width='40%', height='60px')
)

status_output = widgets.Output(layout=widgets.Layout(width='100%', height='100px'))
result_output = widgets.Output(layout=widgets.Layout(width='100%'))

# Enhanced conversion function
def enhanced_voice_conversion(b):
    with status_output:
        status_output.clear_output()
        
        model = model_name.value.strip()
        source_type = input_source.value
        
        if not model:
            print("❌ Please specify a model name")
            return
        
        # Handle different input sources
        input_path = None
        source_title = "Audio File"
        
        if source_type == "file" and audio_file.value:
            # Handle uploaded file
            uploaded_file = list(audio_file.value.keys())[0]
            with open(f"/content/{uploaded_file}", 'wb') as f:
                f.write(audio_file.value[uploaded_file]['content'])
            input_path = f"/content/{uploaded_file}"
            source_title = os.path.splitext(uploaded_file)[0]
            
        elif source_type == "youtube":
            # Handle YouTube URL
            url = youtube_url.value.strip()
            if not url:
                print("❌ Please provide a YouTube URL")
                return
            
            print("📥 Extracting audio from YouTube...")
            try:
                # Use yt-dlp to extract audio
                cmd = [
                    'yt-dlp',
                    '--extract-audio',
                    '--audio-format', 'wav',
                    '--audio-quality', '192K',
                    '-o', '/content/youtube_%(title)s.%(ext)s',
                    url
                ]
                result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
                
                if result.returncode == 0:
                    # Find the downloaded file
                    import glob
                    wav_files = glob.glob('/content/youtube_*.wav')
                    if wav_files:
                        input_path = wav_files[0]
                        source_title = "YouTube Audio"
                    else:
                        print("❌ Failed to find downloaded audio file")
                        return
                else:
                    print(f"❌ YouTube extraction failed: {result.stderr}")
                    return
            except Exception as e:
                print(f"❌ YouTube processing error: {e}")
                return
        
        if not input_path:
            print("❌ No valid input provided")
            return
        
        print(f"🎵 Starting conversion...")
        print(f"📄 Model: {model}")
        print(f"🎯 Source: {source_type} ({source_title})")
        print(f"🎼 F0 Method: {f0_method.value}")
        
        # Prepare output path
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        base_name = f"{source_title}_{model}_{timestamp}"
        if len(base_name) > 50:
            base_name = f"RVC_Conversion_{timestamp}"
        
        output_path = f"/content/RVC/output/RVC_output/{base_name}.{output_format.value}"
        
        # Create output directory
        os.makedirs("/content/RVC/output/RVC_output", exist_ok=True)
        
        try:
            # Run enhanced inference
            cmd = [
                'python3', '-c', f'''
import sys
sys.path.append('/content/RVC')
from rvc.infer.infer import rvc_infer
import os

try:
    result = rvc_infer(
        rvc_model="{model}",
        input_path="{input_path}",
        f0_method="{f0_method.value}",
        f0_min=50,
        f0_max=1100,
        hop_length=128,
        rvc_pitch={pitch_shift.value},
        protect={protect.value},
        index_rate={index_rate.value},
        volume_envelope=1.0,
        output_format="{output_format.value}"
    )
    print("SUCCESS: Conversion completed")
except Exception as e:
    print(f"ERROR: {{e}}")
    sys.exit(1)
'''
            ]
            
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
            
            if result.returncode == 0:
                print("✅ Voice conversion completed successfully!")
                
                # Try to find the output file
                import glob
                output_files = glob.glob(f"/content/RVC/output/RVC_output/{base_name}*.{output_format.value}")
                
                if output_files:
                    actual_output = output_files[0]
                    with result_output:
                        result_output.clear_output()
                        print(f"🎧 Audio file: {os.path.basename(actual_output)}")
                        display(Audio(actual_output, rate=44100))
                else:
                    print("⚠️  Output file not found, but conversion reported success")
            else:
                print(f"❌ Conversion failed: {result.stderr}")
                if result.stdout:
                    print(f"Output: {result.stdout}")
        
        except subprocess.TimeoutExpired:
            print("⏰ Conversion timed out")
        except Exception as e:
            print(f"❌ Conversion error: {e}")

convert_button.on_click(enhanced_voice_conversion)

# Display the interface
display(widgets.VBox([
    widgets.HTML("<h2>🎵 Enhanced Voice Conversion</h2>"),
    widgets.HBox([model_name, input_source]),
    widgets.HBox([audio_file, youtube_url]),
    widgets.HBox([f0_method, pitch_shift]),
    widgets.HBox([index_rate, protect]),
    with_accordion,
    widgets.HBox([output_format, enable_poluvr, enable_custom_kernels, mixed_precision]),
    convert_button,
    status_output,
    result_output
]))

In [ ]:
#@title <big> 🌐 **Launch Enhanced Gradio Demo**

%cd /content/RVC

import threading
import time
from IPython.display import display, HTML, clear_output
import socket

def find_free_port():
    """Find a free port to use for the Gradio server"""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        s.listen(1)
        port = s.getsockname()[1]
    return port

def run_gradio_server():
    """Run the Gradio server in a separate thread"""
    try:
        port = find_free_port()
        print(f"🚀 Starting Enhanced Gradio Demo on port {port}...")
        
        # Import and run the enhanced demo
        from gradio_demo import create_enhanced_gradio_interface
        
        demo = create_enhanced_gradio_interface()
        
        demo.launch(
            server_name="0.0.0.0",
            server_port=port,
            share=False,
            debug=False,
            show_error=True,
            show_tips=True,
            height=800,
            title="Enhanced RVC - PolUVR + YTDLP + Custom Kernels",
            quiet=False
        )
        
    except Exception as e:
        print(f"❌ Failed to start Gradio server: {e}")
        print("🔄 Trying alternative startup method...")
        
        try:
            # Alternative: Run the basic inference interface
            import sys
            sys.path.append('/content/RVC')
            
            # Simple fallback demo
            print("📝 Starting simple Gradio interface...")
            exec(open('gradio_demo.py').read())
            
        except Exception as e2:
            print(f"❌ Alternative also failed: {e2}")

# Start the server in a background thread
gradio_thread = threading.Thread(target=run_gradio_server, daemon=True)
gradio_thread.start()

# Wait a moment for the server to start
time.sleep(3)

print("✅ Enhanced Gradio Demo is starting...")
print("")
print("🌟 Features Available:")
print("   🎛️  PolUVR for source separation")
print("   🌐 YTDLP for YouTube audio extraction")
print("   ⚡ Custom CUDA kernels for optimization")
print("   🎭 Multiple voice models support")
print("   🎼 Advanced F0 extraction methods")
print("   📊 Real-time audio visualization")
print("")
print("📱 Access the demo at the URL provided above")
print("💡 Use the '🔗 Share' button if needed")

In [ ]:
#@title <big> 🔧 **Performance Monitoring & Custom Kernels Test**

%cd /content/RVC

import torch
import time
import psutil
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt

def get_system_info():
    """Get system performance information"""
    info = {
        'CUDA Available': torch.cuda.is_available(),
        'CUDA Version': torch.version.cuda if torch.cuda.is_available() else 'N/A',
        'PyTorch Version': torch.__version__,
        'Device Count': torch.cuda.device_count() if torch.cuda.is_available() else 0,
        'Current Device': torch.cuda.current_device() if torch.cuda.is_available() else 'CPU',
        'Memory (GB)': round(psutil.virtual_memory().total / (1024**3), 1),
        'CPU Cores': psutil.cpu_count()
    }
    return info

def test_custom_kernels():
    """Test custom kernel performance"""
    results = {}
    
    try:
        from rvc.lib.custom_kernels import CustomCUDAKernels, OptimizedAudioProcessor
        
        print("🔧 Testing Custom CUDA Kernels...")
        
        # Test audio processor
        processor = OptimizedAudioProcessor()
        
        # Create test data
        test_audio = torch.randn(44100 * 5)  # 5 seconds
        if torch.cuda.is_available():
            test_audio = test_audio.cuda()
        
        # Test RMS normalization
        start_time = time.time()
        normalized = processor.optimized_rms_normalization(test_audio)
        end_time = time.time()
        
        results['RMS Normalization'] = {
            'Time (ms)': round((end_time - start_time) * 1000, 2),
            'Status': '✅ Success'
        }
        
        print("✅ Custom kernels test completed")
        
    except Exception as e:
        print(f"❌ Custom kernels test failed: {e}")
        results['Custom Kernels'] = {
            'Error': str(e),
            'Status': '❌ Failed'
        }
    
    return results

def performance_benchmark():
    """Run performance benchmarks"""
    print("⚡ Running Performance Benchmarks...")
    
    benchmarks = {}
    
    # Test tensor operations
    if torch.cuda.is_available():
        device = 'cuda'
        print("🔧 Testing on GPU...")
    else:
        device = 'cpu'
        print("🔧 Testing on CPU...")
    
    # Matrix multiplication test
    size = 1000
    A = torch.randn(size, size, device=device)
    B = torch.randn(size, size, device=device)
    
    # Warmup
    for _ in range(3):
        _ = torch.mm(A, B)
    if device == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    start_time = time.time()
    for _ in range(10):
        C = torch.mm(A, B)
    if device == 'cuda':
        torch.cuda.synchronize()
    end_time = time.time()
    
    avg_time = (end_time - start_time) / 10
    benchmarks['Matrix Multiplication'] = {
        'Avg Time (ms)': round(avg_time * 1000, 2),
        'Size': f'{size}x{size}',
        'Device': device.upper()
    }
    
    return benchmarks

# Display system information
print("🖥️  System Information")
print("=" * 40)
system_info = get_system_info()
for key, value in system_info.items():
    print(f"{key}: {value}")

print("\n" + "="*50)

# Run custom kernels test
kernel_results = test_custom_kernels()

print("\n" + "="*50)

# Run performance benchmarks
benchmarks = performance_benchmark()

print("\n" + "="*50)
print("📊 Performance Results")
print("="*50)

print("\n🔧 Custom Kernels:")
for test_name, result in kernel_results.items():
    print(f"  {test_name}: {result}")

print("\n⚡ Performance Benchmarks:")
for test_name, result in benchmarks.items():
    print(f"  {test_name}: {result}")

# Memory usage
if torch.cuda.is_available():
    print(f"\n💾 GPU Memory:")
    print(f"  Allocated: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
    print(f"  Reserved: {torch.cuda.memory_reserved()/1024**3:.1f} GB")
    print(f"  Max Allocated: {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")

print(f"\n💾 System Memory:")
print(f"  Used: {psutil.virtual_memory().used/1024**3:.1f} GB / {psutil.virtual_memory().total/1024**3:.1f} GB")
print(f"  Available: {psutil.virtual_memory().available/1024**3:.1f} GB")
print(f"  Usage: {psutil.virtual_memory().percent}%")

print("\n✅ Performance monitoring completed!")
print("\n💡 Optimization Recommendations:")

if torch.cuda.is_available():
    print("  ✅ GPU acceleration enabled")
    if torch.cuda.get_device_capability()[0] >= 7:
        print("  ✅ TensorFloat-32 supported (A100/RTX 30xx+)")
    if "cupy" in str(kernel_results):
        print("  ✅ Custom CUDA kernels available")
else:
    print("  ⚠️  No GPU detected, using CPU (slower)")

print("\n🎯 Next Steps:")
print("  1. Install a voice model using the model management section")
print("  2. Use the Enhanced Voice Conversion section")
print("  3. Try the Gradio demo for a visual interface")